In [ ]:
%%configure
{ "vCores": 64 }

# The duckrun arm: DuckDB reads, delta-rs writes

**Hand-written. `build_notebooks.py` does NOT emit this file** -- same as `build_vorder.ipynb`. A
rebuild will not touch it, and an edit here will not be lost.

A third writer. Every other arm in this repo comes from parquet-mr (the Databricks arms) or from
Fabric Spark's V-Order writer; this one is **delta_rs**, via delta-rs, driven by duckrun. It reads
the already-mirrored `tpcds_sf{sf}_default` rows and rewrites them -- no `dsdgen`, no second copy of
the raw data.

## Why a third writer is worth a run

**One: the dictionary claim.** LEARNING.md's sharpest sentence is about a conflict only V-Order was
ever measured to resolve:

> the recipe's 6M row groups and the paper's partition-by-date are in direct conflict over the
> dictionary, and V-Order is what resolves it. The Fabric arm holds 100 % dictionary at the same
> 143k-row geometry where parquet-mr holds ~50 %. *That is the one thing in this whole comparison
> that a non-Fabric writer cannot reproduce.*

delta_rs does not run parquet-mr's `FallbackValuesWriter`. **There is no first-page cost test**: a
column keeps its dictionary until the dictionary itself passes `dictionary_page_size_limit`. So the
`partition` variant below is a direct test of that sentence. If delta_rs holds its dictionary at
143k-row row groups, the sentence is false. If it collapses the same way, the sentence is confirmed
by a writer nobody had asked.

The corollary is already worth writing down: **the recipe's page settings are a parquet-mr
workaround, not a layout property.** `parquet.page.row.count.limit` = 16M and 64 MB pages exist
solely to move that first page far enough out that the dictionary wins the cost test. delta_rs has
no such test, so duckrun's 1 MB / 1M-row pages are the right shape here and the recipe's numbers
would buy nothing but write memory.

**Two: an ordering nobody chose.** Every ordered arm in this repo sorts on one key picked by hand --
the date surrogate, because 23 of the paper's 24 captured queries filter on it. `SORTED BY AUTO` is
duckrun's own recommender, and it picks its key to minimise **modelled in-memory columnar bytes**,
which is a different objective. That makes the `default` arm a real question rather than a copy of
`cluster` with another writer: does a key chosen for size land anywhere near a key chosen for
pruning?

## What geometry this writes, and why nothing here sets it

duckrun's write profile is FIXED. `max_row_group_size` / `target_file_size` are exposed as **dbt
model configs only** -- `session.py` passes neither, so every `conn.sql()` write gets the module
constants in `dbt/adapters/duckrun/engine.py` and `policy.py` verbatim.

As of **duckrun 0.4.68** (released 2026-09-07) that profile is:

| knob | value | against what this project measured |
|---|---|---|
| `max_row_group_size` | 4,000,000 rows | **inside the measured 2M-5M optimum.** Nothing to tune -- that it lands there is the finding |
| `target_file_size` | 256 MB | rolls the file mid-row-group, so every file ends on a truncated group |
| `dictionary_page_size_limit` | 32 MB | the only fallback exit delta_rs has. A 4M-row INT32 key needs ~16 MB, so it fits |
| `data_page_size_limit` | 1 MB | |
| `data_page_row_count_limit` | 1,000,000 | |
| compression | **SNAPPY** | **a stated confound -- see below** |

**These numbers move between duckrun releases, so the setup cell PRINTS the profile the installed
version will actually use, and that print -- not this table -- is the record of what the arm was
written under.** (`CHANGELOG.md` is no help in dating them: everything since 0.4.31 is filed under
one `[Unreleased]` heading.)

**The row-group ceiling is a ceiling, not a size**, and at this scale the 256 MB roll is close
enough to it that both fire. `store_sales` at ~36 B/row puts ~7.1M rows in a 256 MB file, so the 4M
ceiling cuts first and leaves a ~3.1M tail group; `catalog_sales` at ~57 B/row puts ~4.5M there, so
the ceiling cuts at 4M and leaves a ~0.5M tail -- **below Direct Lake's 1M segment floor**. So the
group size that lands is bytes / row width, per table, which is exactly what the Databricks recipe's
three row-denominated numbers exist to prevent. The last cell reports `avg_row_group` per table
rather than predicting it.

**SNAPPY is a confound and it is not fixable from here.** Every other arm is ZSTD (DBR 19 stamps it;
`build_vorder` sets it explicitly). `engine._writer_properties()` hard-codes `compression="SNAPPY"`
and the notebook API cannot override it. The *dictionary* question above survives that -- encoding
is classified from the footer's `encodings`, not from bytes -- but any **size** or **cold-load**
comparison against another arm does not. Read this arm's bytes against another arm's only after that
is fixed, in duckrun or with explicit `WriterProperties`.

`checkpointPolicy` needs no pin here: delta-rs has no v2 checkpoint to turn on. The last cell reads
the protocol back rather than assuming it.

## The two variants

`variant` is the only thing that changes the write.

| variant | schema | clause | what it isolates |
|---|---|---|---|
| `default` | `tpcds_sf{sf}_duckdb` | `SORTED BY AUTO`, **every table** | what duckrun does when you let it decide: it profiles the data and picks the ORDER BY itself |

`SORTED BY AUTO` goes on the dimensions too, because the arm is "let duckrun decide" and AUTO writes
unsorted when nothing pays off -- it self-limits on a 20-row table. `PARTITIONED BY` is facts-only:
the dimensions have no date key.

Either way the sort **survives to the files**: delta-rs has no Optimized Writes exchange to delete
it, which is the trap measured on the Databricks side, where Spark drops an `orderBy` when optimized
writes are on. And it is a GLOBAL order, not a per-file or per-row-group one, so each row group's
min/max range is disjoint from its neighbours' -- the property that lets a reader skip groups.

### What AUTO picks, and why the notebook records it

The statement below is a bare `SELECT * FROM <one delta table>`, which is duckrun's best-supported
AUTO path: it profiles the source from its **Delta log** (null shares, NDV caps) rather than from a
bare relation scan. The picker takes the coarsest temporal column first, then ascending cardinality
up to 4 dimension columns, skips functionally dependent columns, stops at the grain, and lets
measures take only tail slots below the whole key.

**`ss_sold_date_sk` is an INT surrogate, not a temporal type**, so it probably does not win the lead
slot on "coarsest date" -- it competes on cardinality (~1,823 distinct) against genuinely coarser
columns like `ss_quantity` (100) and `ss_store_sk` (402). LEARNING.md's rule is that **only the
first sort key eliminates row groups**. So the prediction, stated before the run: `default` gives up
the date-filtered queries `cluster` wins, and buys size and cold load instead. Either result is a
finding -- but only if the key is written down, which is why the write loop captures duckrun's
advisory and the report prints the key it chose.

**`default` is the expensive variant.** AUTO is a global `ORDER BY` over the whole table, staged in
DuckDB and spilled to `temp_directory`, so **local disk on the node is the ceiling** -- 288M rows for
`store_sales` at SF100. And above `SUBSTRATE_CAP` (30M rows) duckrun profiles a ~30M-row
`hash(row) % K` substrate and the write then re-reads the ORIGINAL source, so each fact is read from
the mirror **twice**. At SF10 both facts are under the cap: every row is profiled and the source is
staged once. **Validate at SF10 before committing an SF100 run.**

One thing to watch in the output: duckrun rewrites `DECIMAL(p > 18)` to `DECIMAL(18, s)`, because
delta_rs never dictionary-encodes a 16-byte FIXED_LEN_BYTE_ARRAY. TPC-DS money columns are
`DECIMAL(7,2)`, so nothing should fire -- and duckrun prints one advisory line per wide-decimal
column, so silence is the confirmation.

In [ ]:
!pip install -q duckrun --upgrade
notebookutils.session.restartPython()

In [ ]:
sf = 100
# "default"   = SORTED BY AUTO on every table: duckrun profiles the data and picks the ORDER BY
#               itself. The key it chose is printed per table and lands in the report.
# "sorted"    = SORTED BY the DATE key on the facts -- the same key the Databricks `cluster` arm
#               orders on, so this is that arm's ordering under a different writer.
variant = "default"
src_schema = ""      # empty -> tpcds_sf{sf}_default in the mirrored catalog

In [ ]:
import contextlib
import io
import time

import duckrun
import notebookutils

MIRROR = "01b539f3-4a9d-45ef-b1ef-0ba59552eb21"   # Mirrored Azure Databricks catalog item
VORDER_LH = "tpcds_vorder"                        # the arm is the SCHEMA suffix, not the item

sf = int(sf)
assert variant in ("default", "sorted"), \
    f"variant must be 'default' or 'sorted', not {variant!r}"

# The same source `build_vorder` reads, and for the same reason: the UNSORTED Databricks arm is the
# one that imposes no ordering of its own on what the next writer then does.
SRC_SCHEMA = src_schema or f"tpcds_sf{sf}_default"
# One schema per variant. Two layouts wearing one name is the failure mode this project exists to
# avoid, so the variant is in the name and a name is never reused.
SCHEMA = {"default":   f"tpcds_sf{sf}_duckdb",
          "sorted":    f"tpcds_sf{sf}_ducksort"}[variant]

# Table 4.6.1's first column: the date surrogate key -- the PARTITION key, and the key every other
# ordered arm here uses, because 23 of the paper's 24 captured queries filter on it. `sorted` uses
# it; the `default` arm does NOT -- AUTO picks its own, which is that arm's whole point.
FACTS = {"store_sales": "ss_sold_date_sk", "catalog_sales": "cs_sold_date_sk"}
DIMS = ["catalog_page", "customer_address", "customer_demographics", "date_dim",
        "item", "promotion", "ship_mode", "store"]
TABLES = DIMS + list(FACTS)

ws_id = notebookutils.runtime.context["currentWorkspaceId"]
# GUIDs, never friendly names: this tenant has OneLake friendly-name support disabled, and a
# friendly name can trip a delta_scan bug (duckdb-delta#307).
vorder_id = notebookutils.lakehouse.get(VORDER_LH)["id"]

# `schema=` wins over the path, so the root stays .../Tables and discovery is scoped to the ONE
# schema this run writes -- a target schema that does not exist yet discovers nothing, and the
# write creates it. read_only=False is required: duckrun refuses every Delta write by default.
conn = duckrun.connect(f"{ws_id}/{vorder_id}", schema=SCHEMA, read_only=False, name="tgt")
# The mirror, read-only. read_only is PER CATALOG, so this cannot be written to by accident even
# though the session is writable.
conn.attach(f"{ws_id}/{MIRROR}", schema=SRC_SCHEMA, read_only=True, name="src")

print(f"duckrun {duckrun.__version__}")
print(f"SF{sf}:  src.{SRC_SCHEMA}  ->  tgt.{SCHEMA}   (variant={variant})")

# THE GEOMETRY, read from the installed duckrun rather than claimed. Nothing in this notebook sets
# any of it -- max_row_group_size / target_file_size are dbt model configs and the notebook API
# passes neither -- and the constants move between releases (0.4.68 took the ceiling from 6M to 4M).
# So print what will actually be used, and treat this line as the record of what the arm was written
# under. `!pip install --upgrade` above means a re-run months later can write a different geometry
# under the same schema name, and this print is the only thing that would say so.
try:
    from dbt.adapters.duckrun import engine, policy

    print(f"\nwrite profile (duckrun's, fixed): "
          f"max_row_group_size={policy.ROW_GROUP_DEFAULT_ROWS:,} rows  "
          f"target_file_size={policy.DEFAULT_TARGET_FILE_SIZE / 1024 ** 2:,.0f} MB")
    print("writer_properties:", engine._writer_properties())
except Exception as e:                                          # noqa: BLE001
    # Private constants; a rename in a future duckrun must not fail the build, only this report.
    print(f"\n[warn] could not read duckrun's write profile ({e}) -- "
          f"read it off the footers in the last cell instead")

In [ ]:
# NEVER OVERWRITES. A table that already exists is left exactly as it is and the loop moves on, so
# re-running this notebook finishes a partial build instead of redoing an hour of writes. Same rule
# as both other builders. To genuinely rebuild, DROP the schema by hand.
#
# The guard has to be explicit: duckrun's `CREATE TABLE ... AS` is a Delta OVERWRITE, so nothing in
# the statement itself would stop a second run destroying a built arm.
existing = {r[2] for r in conn.sql("SHOW ALL TABLES").fetchall()
            if r[0] == "tgt" and r[1] == SCHEMA}
if existing:
    print(f"{SCHEMA}: {len(existing)} table(s) already present, they will be SKIPPED "
          f"-- drop the schema by hand to force a rebuild", flush=True)


# `SORTED BY ...` is duckrun's CTAS extension and sits between the table name and AS. `default`
# sorts EVERY table (AUTO writes unsorted when nothing pays off, so it self-limits on a 20-row
# dimension); `sorted` touches the FACTS only, because the dimensions have no date key.
def clause_for(table):
    if variant == "default":
        return "SORTED BY AUTO "
    return f"SORTED BY ({FACTS[table]}) " if table in FACTS else ""


# AUTO's chosen key is printed by duckrun, not returned, so it is captured here and reported below.
# Without it the arm is uninterpretable: "sorted by something" is not a measurement. The captured
# text is echoed straight back, so the cell output is exactly what it would have been.
sort_key = {}


def run_write(table):
    stmt = f"CREATE TABLE {SCHEMA}.{table} {clause_for(table)}AS SELECT * FROM src.{SRC_SCHEMA}.{table}"
    if variant != "default":
        conn.sql(stmt)
        return
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            conn.sql(stmt)
    finally:
        out = buf.getvalue()
        if out:
            print(out, end="", flush=True)
    # duckrun's advisory ends with the ORDER BY it chose, or says nothing paid off.
    picked = [ln.strip() for ln in out.splitlines() if ln.strip().startswith("ORDER BY")]
    sort_key[table] = picked[-1] if picked else "(unsorted -- nothing paid off)"


# FACTS FIRST. The expensive, load-bearing work goes before anything that can fail cheaply -- on
# 2026-09-08 the vorder notebook's maintenance loop ran dimensions first, died on the first one and
# never reached the facts.
written, skipped = [], []
for t in list(FACTS) + DIMS:
    t0 = time.time()
    if t in existing:
        skipped.append(t)
        print(f"{t}: exists, skipped", flush=True)
        continue
    run_write(t)
    written.append(t)
    print(f"{t}: written {clause_for(t) or '(no ordering)'} in {time.time() - t0:,.0f}s"
          + (f"  key: {sort_key[t]}" if t in sort_key else ""), flush=True)

print()
print(f"{len(written)} written, {len(skipped)} skipped, {len(TABLES)} expected")
assert len(written) + len(skipped) == len(TABLES), "a table is neither written nor skipped"

In [ ]:
# What landed, next to what it was read from. `get_stats` reads the Delta log for the live file list
# and then the parquet footers, so `total_rows` is the LOGICAL row count (matching COUNT(*)) and
# `avg_row_group` is the PHYSICAL segment size -- one parquet row group is one VertiPaq segment.
import pandas as pd

tgt = conn.get_stats(SCHEMA).df()
src = conn.get_stats(SRC_SCHEMA).df()

cmp = (tgt.merge(src[["table", "total_rows", "num_files", "num_row_groups", "avg_row_group",
                      "size_mb", "compression"]],
                 on="table", suffixes=("", "_src"))
          [["table", "total_rows", "total_rows_src", "num_files", "num_files_src",
            "num_row_groups", "num_row_groups_src", "avg_row_group", "avg_row_group_src",
            "size_mb", "size_mb_src", "compression", "compression_src"]]
          .sort_values("total_rows", ascending=False))
# The key AUTO chose, next to the geometry it produced. A table skipped on this run has no entry --
# its key is in the output of the run that wrote it, not here.
if sort_key:
    cmp.insert(1, "sort_key", cmp["table"].map(sort_key).fillna("(skipped this run)"))
display(cmp)

# The one assertion that catches a silent partial write. Both counts come from a Delta log, so it
# costs nothing.
lost = cmp[cmp.total_rows != cmp.total_rows_src]
assert lost.empty, f"row count differs from the source on {list(lost['table'])}"

# Direct Lake's usable window is 1M-16M rows per segment; the measured optimum is 2M-5M. Reported
# rather than asserted: duckrun's geometry is its own and this notebook sets none of it.
RG_LO, RG_HI = 1_000_000, 16_000_000
print()
for r in cmp.itertuples():
    if r.table in FACTS:
        state = "in window" if RG_LO <= r.avg_row_group <= RG_HI else "OUTSIDE the 1M-16M window"
        print(f"{r.table:<16} avg row group {r.avg_row_group:>12,.0f} rows  {state}")

# Protocol, read back rather than assumed. Direct Lake cannot read a v2 checkpoint AT ALL. delta-rs
# has no v2 checkpoint to turn on, but this is the first table this repo has written with it.
from deltalake import DeltaTable

dt = DeltaTable(f"{conn.root_path}/{SCHEMA}/store_sales", storage_options=conn.storage_options)
cfg = dict(dt.metadata().configuration or {})
print()
print("protocol:", dt.protocol())
print("configuration:", cfg or "(none)")
assert cfg.get("delta.checkpointPolicy", "classic") == "classic", \
    "checkpointPolicy is not classic -- Direct Lake cannot read a v2 checkpoint"